# Notebook 01: Data formats and AI-assisted coding

**Introduction to AI-Assisted Research Methods and Data Analysis** · Day 2, block 1

NexusHealth · Adamu Onu

Clinical data arrive in many shapes: CSV, Excel, SPSS, Stata and REDCap exports, wide or long. In this notebook you will read each one, and learn to use an AI coding assistant without handing over your judgement.

> **Teaching data only.** Every patient in this course is synthetic. The associations are simulated and are not clinical evidence. Never paste identifiable patient data, confidential manuscripts or restricted full text into a public AI tool.

### How to use this notebook

1. Run cells from top to bottom with **Shift + Enter**.
2. Before you run a cell, say out loud (or write) what you expect it to do. After it runs, check whether you were right. We call this *predict, run, inspect*.
3. Cells marked **Exercise** have gaps for you to fill. Cells marked **AI assist** suggest a prompt for GitHub Copilot Chat (or your own assistant).
4. If things get tangled, use **Restart** then **Run All** from the notebook toolbar.

In [ ]:
suppressPackageStartupMessages({
  library(tidyverse)
})

# Find the course folder, wherever this notebook lives inside it.
ROOT <- normalizePath(getwd())
while (!dir.exists(file.path(ROOT, "data", "raw")) && dirname(ROOT) != ROOT) {
  ROOT <- dirname(ROOT)
}
stopifnot("Course folder not found. Open the whole course folder in VS Code." =
            dir.exists(file.path(ROOT, "data", "raw")))

DATA_RAW       <- file.path(ROOT, "data", "raw")
DATA_PROCESSED <- file.path(ROOT, "data", "processed")
DATA_REFERENCE <- file.path(ROOT, "data", "reference")
OUT_TABLES     <- file.path(ROOT, "outputs", "tables")
OUT_FIGURES    <- file.path(ROOT, "outputs", "figures")
for (folder in c(DATA_PROCESSED, OUT_TABLES, OUT_FIGURES)) dir.create(folder, recursive = TRUE, showWarnings = FALSE)

options(repr.plot.width = 7, repr.plot.height = 4.5, pillar.width = 120)
theme_set(theme_minimal(base_size = 12))
cat("Course folder found:", basename(ROOT), "\n")

# Part A: Data formats

| Format | Extension | Typical source | Watch out for |
|---|---|---|---|
| CSV | `.csv` | Exports from almost anything | No types stored; missing-value codes; date formats |
| Excel | `.xlsx` | Hand-entered audit data | Title rows, merged cells, several sheets, dates turned into numbers |
| SPSS | `.sav` | Survey and psychology research | Numbers with value labels (1 = Female) |
| Stata | `.dta` | Epidemiology and economics | Value labels, as with SPSS |
| REDCap export | `.csv` + codebook | Research databases | Coded values whose meaning is in a separate file |

A good habit: **keep the raw file untouched**. Read it, clean a copy in code, and save the clean version under a new name.

## A1. CSV

You met this in Notebook 00. CSV files store everything as text, so the reading software has to guess types. Notice the different ways missing values were typed.

In [ ]:
raw_text <- read_csv(file.path(DATA_RAW, "primary_care_raw.csv"),
                     col_types = cols(.default = "c"), na = character(), trim_ws = FALSE)
raw_text |> filter(is.na(suppressWarnings(as.numeric(bmi)))) |> count(bmi)

Four different spellings of *missing* (`blank`, `NA`, `N/A`, `Unknown`). We will fix these on the cleaning day. For now, notice that you only see them if you read the file as text first.

## A2. Excel

Our Excel export has two title rows and a blank row above the real header, and a second sheet holding the dictionary. Look at the first few rows as they really are.

In [ ]:
xlsx_path <- file.path(ROOT, "data", "formats", "primary_care_export.xlsx")
print(readxl::excel_sheets(xlsx_path))
readxl::read_excel(xlsx_path, sheet = "visits", col_names = FALSE, n_max = 5)

Skip the three rows above the header to get a proper table:

In [ ]:
visits_xlsx <- readxl::read_excel(xlsx_path, sheet = "visits", skip = 3, trim_ws = FALSE)
dim(visits_xlsx)
head(visits_xlsx, 3)

## A3. SPSS and Stata files with value labels

SPSS and Stata store categories as numbers and attach **labels** to them. If you ignore the labels you might analyse `sex` as if 2 were twice as much as 1.

In [ ]:
library(haven)
sav <- read_sav(file.path(ROOT, "data", "formats", "primary_care_subset.sav"))
print(attr(sav$sex, "labels"))
print(attr(sav$bmi, "label"))
head(sav, 3)

Convert the coded numbers into labelled categories:

In [ ]:
sav_labelled <- as_factor(sav)
sav_labelled |> select(patient_id, clinic, sex, smoking_status, diabetes) |> head(3)

Stata files work the same way:

In [ ]:
dta <- read_dta(file.path(ROOT, "data", "formats", "primary_care_subset.dta")) |> as_factor()
head(dta, 3)

## A4. REDCap-style export

Many research databases export coded numbers in one file and the meaning of each code in a separate codebook. Always read both.

In [ ]:
redcap <- read_csv(file.path(ROOT, "data", "formats", "redcap_style_export.csv"), show_col_types = FALSE)
codebook <- read_csv(file.path(ROOT, "data", "formats", "redcap_style_codebook.csv"), show_col_types = FALSE)
print(codebook)
head(redcap, 3)

## A5. Wide and long formats

When a patient has repeated measurements you can store them **long** (one row per reading) or **wide** (one row per patient, one column per reading). Long format suits most analysis and plotting. Wide format suits some summaries and is how many spreadsheets look.

In [ ]:
long <- read_csv(file.path(ROOT, "data", "formats", "bp_readings_long.csv"), show_col_types = FALSE)
cat("Long format:", dim(long), "\n")
print(head(long, 6))

wide <- long |>
  select(patient_id, reading, systolic_bp) |>
  pivot_wider(names_from = reading, values_from = systolic_bp, names_prefix = "sbp_reading_")
cat("Wide format:", dim(wide), "\n")
head(wide, 3)

## Exercise 1A: Same patients, different files

Count the number of visits per clinic in the **Excel** file. Then check your answer against the same count from the **CSV** file. They should match exactly (before any cleaning, the clinic labels are still messy, which is fine for this check).

In [ ]:
# Count clinics in the Excel data
xlsx_counts <- ...

# Count clinics in the CSV data
csv_counts <- ...

# Do they match?
...

# Part B: Working with an AI coding assistant

This course uses **GitHub Copilot** inside VS Code. Copilot has a free plan, and verified students and teachers can apply for extra access through GitHub Education (check current terms). If you already pay for another assistant (for example Gemini Code Assist, Claude, ChatGPT or Cursor) you can use it instead. If you cannot install software at all, **Google Colab** runs notebooks in a browser and has Gemini built in.

Three ways to use Copilot in a notebook:

1. **Inline suggestions.** Type a comment such as `# count visits per clinic` and pause. Grey text appears; press **Tab** to accept it or keep typing to ignore it.
2. **Copilot Chat.** Open the chat panel and ask questions in plain language.
3. **Explain.** Select some code, right-click, and choose **Copilot > Explain**. Use this whenever you do not understand a line, including lines that we wrote.

> **Rule of thumb:** AI code that runs is not the same as AI code that is right. You are responsible for every number that leaves your notebook.

### A five-part prompt for analysis code

Vague prompts get vague code. Give the assistant the same information you would give a colleague:

| Part | What to include | Example |
|---|---|---|
| **Goal** | The research task, in one sentence | Count visits per clinic |
| **Data** | File, variable names, types (never real patient values) | `primary_care_raw.csv`; `clinic` is text |
| **Constraints** | Language, packages, style | R; tidyverse; short and commented |
| **Output** | What you want back | A table with counts and percentages |
| **Check** | How you will know it is right | Counts must add up to the number of rows |


**AI assist (try it now).** Paste this into Copilot Chat:

> *Goal:* summarise systolic blood pressure. *Data:* a tibble called `raw` loaded from `primary_care_raw.csv`; `systolic_bp` should be numeric (mmHg). *Constraints:* R with the tidyverse, short and commented. *Output:* the number of non-missing values, mean, median, minimum and maximum. *Check:* tell me what to look at to know whether the result is plausible.

Paste the code it suggests into the empty cell below and run it.

In [ ]:
raw <- read_csv(file.path(DATA_RAW, "primary_care_raw.csv"), show_col_types = FALSE)
# Paste the AI-suggested code below this line

## Exercise 1B: Audit some AI-generated code

A colleague asked an assistant for *the average systolic BP*, and got the code below. It runs without an error. Run it, then decide whether you would trust the answer.

In [ ]:
# AI-generated code (do not edit this cell)
sbp <- suppressWarnings(as.numeric(raw$systolic_bp))
cat("Average systolic BP:", round(mean(sbp, na.rm = TRUE), 1), "mmHg\n")

Use the checklist to investigate. Write code in the cell below.

* **Denominator:** how many values went into the mean, and how many were silently dropped?
* **Plausibility:** what are the minimum and maximum? Do they make clinical sense?
* **Duplicates:** does any patient count twice?
* **Explain:** ask Copilot to explain what `errors="coerce"` (Python) or `as.numeric` with `suppressWarnings` (R) does to text like `Unknown`.

In [ ]:
# Investigate the AI-generated average here
...

**Would you report the AI's number? Why or why not?** Write two or three sentences.

**Your answer:** _Double-click here and write your answer._

## B2. Your AI-use log

For the capstone you will keep a short log of how you used AI. One row per meaningful use is enough. The template is in `templates/AI_Use_Log_Template.docx`, and a CSV version is created below in `outputs/logs/ai_use_log.csv`. Add a row for the exercise you just did.

In [ ]:
log_path <- file.path(ROOT, "outputs", "logs", "ai_use_log.csv")
dir.create(dirname(log_path), recursive = TRUE, showWarnings = FALSE)
if (!file.exists(log_path)) {
  write_csv(tibble(date = character(), tool = character(), task = character(),
                   prompt_summary = character(), what_i_used = character(),
                   how_i_verified = character(), changes_i_made = character()), log_path)
}

entry <- tibble(
  date = as.character(Sys.Date()),
  tool = "GitHub Copilot Chat",
  task = "Summarise systolic BP",
  prompt_summary = "Five-part prompt: goal, data, constraints, output, check",
  what_i_used = "Summary code (not the result)",
  how_i_verified = "Checked denominator, min/max, duplicates",
  changes_i_made = "Did not report the mean until data are cleaned"
)
log <- bind_rows(read_csv(log_path, col_types = cols(.default = "c")), entry)
write_csv(log, log_path)
tail(log)

## Key points

* Read unfamiliar files as text first, so hidden codes and odd entries become visible.
* Labelled formats (SPSS, Stata, REDCap) store numbers; the meaning lives in labels or a codebook.
* Long format is usually the right shape for analysis.
* Give the AI a goal, the data structure, constraints, the output you want and how you will check it.
* Never accept a number because the code ran. Check the denominator, the range and the duplicates.